# Heteroskedasticity of unknown form: Lasso-FGLS (González-Coya & Perron, 2024)

$y = 1 + x + \sqrt{v(x)}\,\varepsilon$ with $x \sim U[1,4]$ and $v(x) = x^2$.

In [ ]:
import warnings

import numpy as np
import pandas as pd

import pyfgls as pf

warnings.simplefilter("ignore", pf.exceptions.PyFGLSWarning)


In [ ]:
from pyfgls.simulate import hetero_dgp

sim = hetero_dgp(400, skedastic='x2', random_state=1)
df = pd.DataFrame({'y': sim.y, 'x': sim.X[:, 1]})
res = pf.LassoFGLS.from_formula('y ~ x', data=df, random_state=0).fit()
print(res.summary())

The adaptive Lasso picks among $\{x, x^2, \log x, (\log x)^2, \cos x, \cos 2x\}$:

In [ ]:
res.skedastic_coef.round(4)

Against OLS with HC3 standard errors, and the alternative skedastic estimators:

In [ ]:
ols = pf.LassoFGLS.from_formula('y ~ x', data=df, skedastic='none').fit()
svr = pf.LassoFGLS.from_formula('y ~ x', data=df, skedastic='svr', random_state=0).fit()
wls = pf.LassoFGLS.from_formula('y ~ x', data=df, skedastic='wls_s1', z=['x']).fit()
pf.compare([ols, wls, svr, res], names=['OLS', 'WLS-S1', 'SVR', 'Lasso']).round(4)

In [ ]:
from pyfgls.plotting import plot_intervals

plot_intervals([ols, res], labels=['OLS (HC3)', 'Lasso-FGLS']);

Under homoskedasticity the Lasso typically selects nothing and FGLS equals OLS:

In [ ]:
h = hetero_dgp(400, skedastic='homoskedastic', random_state=2)
r0 = pf.LassoFGLS(h.y, h.X, random_state=0).fit()
r0.selected_z, r0.df_hat